In the final utility-based experiment, LLM agents negotiated over the full set of Negochat issues using private utility tables. This allows agreements to be evaluated not only by explicit acceptance, but also by whether the final package is beneficial for both agents.


In [ ]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,

In [ ]:
!pip install ollama

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [ ]:
!ollama pull llama3.2:3b
model_name = "llama3.2:3b"

In [ ]:
import os

REPO_URL = "https://github.com/simona-wang/negotiation_arena.git"
PROJECT_DIR = "/content/negotiation_arena"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
import re
import json
import pandas as pd
import ollama
from tqdm import tqdm

In [ ]:
ISSUES = {
    "job_description": [
        "Programmer",
        "Project Manager",
        "Team Manager"
    ],
    "salary": [
        "75000",
        "80000",
        "85000",
        "90000"
    ],
    "working_hours": [
        "8",
        "9",
        "10"
    ],
    "pension_fund": [
        "0%",
        "10%",
        "20%"
    ],
    "leased_car": [
        "No",
        "Yes"
    ],
    "promotion_possibilities": [
        "Slow",
        "Moderate",
        "Fast"
    ]
}

In [ ]:
candidate_utility_table = {
    "job_description": {
        "Programmer": 1,
        "Project Manager": 3,
        "Team Manager": 4
    },
    "salary": {
        "75000": 0,
        "80000": 1,
        "85000": 3,
        "90000": 4
    },
    "working_hours": {
        "8": 4,
        "9": 2,
        "10": 0
    },
    "pension_fund": {
        "0%": 0,
        "10%": 2,
        "20%": 4
    },
    "leased_car": {
        "No": 0,
        "Yes": 2
    },
    "promotion_possibilities": {
        "Slow": 0,
        "Moderate": 2,
        "Fast": 4
    }
}

In [ ]:
employer_utility_table = {
    "job_description": {
        "Programmer": 4,
        "Project Manager": 2,
        "Team Manager": 1
    },
    "salary": {
        "75000": 4,
        "80000": 3,
        "85000": 2,
        "90000": 0
    },
    "working_hours": {
        "8": 0,
        "9": 2,
        "10": 4
    },
    "pension_fund": {
        "0%": 4,
        "10%": 2,
        "20%": 0
    },
    "leased_car": {
        "No": 2,
        "Yes": 0
    },
    "promotion_possibilities": {
        "Slow": 4,
        "Moderate": 2,
        "Fast": 0
    }
}

In [ ]:
def compute_utility(package, utility_table):
    total = 0
    max_total = 0

    for issue, values in utility_table.items():
        max_total += max(values.values())

        selected_value = package.get(issue)

        if selected_value in values:
            total += values[selected_value]

    normalized_utility = total / max_total if max_total > 0 else 0

    return normalized_utility

In [ ]:
example_package = {
    "job_description": "Project Manager",
    "salary": "85000",
    "working_hours": "9",
    "pension_fund": "10%",
    "leased_car": "Yes",
    "promotion_possibilities": "Moderate"
}

print("Candidate utility:", compute_utility(example_package, candidate_utility_table))
print("Employer utility:", compute_utility(example_package, employer_utility_table))

Candidate utility: 0.6363636363636364
Employer utility: 0.45454545454545453


In [ ]:
def build_agent_prompt(role, style, utility_table):
    return f"""
You are the {role} in a job contract negotiation.

You are negotiating a full job contract package with these issues:

- job_description: Programmer, Project Manager, Team Manager
- salary: 75000, 80000, 85000, 90000
- working_hours: 8, 9, 10
- pension_fund: 0%, 10%, 20%
- leased_car: No, Yes
- promotion_possibilities: Slow, Moderate, Fast

Your private utility table is:

{json.dumps(utility_table, indent=2)}

Negotiation style: {style}

Rules:
- Choose exactly ONE value for each issue.
- Do NOT copy the list of possible values.
- Do NOT write values with slashes such as "75000 / 80000 / 85000 / 90000".
- If you accept the previous package, set "decision": "accept".
- If you continue negotiating, propose a complete package.
- Do not simulate both speakers.
- Reply only as {role}.

Reply ONLY with a valid JSON object in this exact format:

{{
  "message": "your short negotiation message",
  "job_description": "Programmer",
  "salary": "85000",
  "working_hours": "9",
  "pension_fund": "10%",
  "leased_car": "Yes",
  "promotion_possibilities": "Moderate",
  "decision": "continue"
}}
"""

In [ ]:
def generate_response(system_prompt, user_message):
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )

    return response["message"]["content"].strip()

In [ ]:
def parse_package_response(text):
    default = {
        "message": None,
        "job_description": None,
        "salary": None,
        "working_hours": None,
        "pension_fund": None,
        "leased_car": None,
        "promotion_possibilities": None,
        "decision": None
    }

    # Try to extract JSON from the model output
    try:
        start = text.find("{")
        end = text.rfind("}") + 1

        if start != -1 and end != -1:
            json_text = text[start:end]
            parsed = json.loads(json_text)

            for key in default:
                if key in parsed:
                    default[key] = str(parsed[key]).strip()

            if default["decision"] is not None:
                default["decision"] = default["decision"].lower()

            return default

    except Exception:
        pass

    return default

In [ ]:
def extract_package(parsed):
    return {
        "job_description": parsed["job_description"],
        "salary": parsed["salary"],
        "working_hours": parsed["working_hours"],
        "pension_fund": parsed["pension_fund"],
        "leased_car": parsed["leased_car"],
        "promotion_possibilities": parsed["promotion_possibilities"]
    }

In [ ]:
VALID_VALUES = {
    "job_description": ["Programmer", "Project Manager", "Team Manager"],
    "salary": ["75000", "80000", "85000", "90000"],
    "working_hours": ["8", "9", "10"],
    "pension_fund": ["0%", "10%", "20%"],
    "leased_car": ["No", "Yes"],
    "promotion_possibilities": ["Slow", "Moderate", "Fast"]
}


def is_valid_package(package):
    for issue, valid_values in VALID_VALUES.items():
        if package.get(issue) not in valid_values:
            return False
    return True

In [ ]:
MIN_CANDIDATE_UTILITY = 0.55
MIN_EMPLOYER_UTILITY = 0.55

In [ ]:
def is_mutually_acceptable(package):
    candidate_u = compute_utility(package, candidate_utility_table)
    employer_u = compute_utility(package, employer_utility_table)

    return (
        candidate_u >= MIN_CANDIDATE_UTILITY
        and employer_u >= MIN_EMPLOYER_UTILITY
    )

In [ ]:
def run_utility_negotiation_simulation(
    scenario_id,
    condition_name,
    candidate_style,
    employer_style,
    run_id,
    max_turns=10
):
    conversation_log = []

    candidate_prompt = build_agent_prompt(
        role="Candidate",
        style=candidate_style,
        utility_table=candidate_utility_table
    )

    employer_prompt = build_agent_prompt(
        role="Employer",
        style=employer_style,
        utility_table=employer_utility_table
    )

    current_message = """
{
  "message": "I would like to discuss a full job contract package.",
  "job_description": "Project Manager",
  "salary": "90000",
  "working_hours": "8",
  "pension_fund": "20%",
  "leased_car": "Yes",
  "promotion_possibilities": "Fast",
  "decision": "continue"
}
"""

    outcome = None
    compatible_package_seen = False
    best_candidate_utility = 0
    best_employer_utility = 0

    last_valid_package = None
    last_candidate_utility = 0
    last_employer_utility = 0

    for turn in range(max_turns):

        # Employer turn
        employer_text = generate_response(
            employer_prompt,
            current_message
        )

        parsed_employer = parse_package_response(employer_text)
        employer_package = extract_package(parsed_employer)

        if is_valid_package(employer_package):
            candidate_u = compute_utility(employer_package, candidate_utility_table)
            employer_u = compute_utility(employer_package, employer_utility_table)

            last_valid_package = employer_package
            last_candidate_utility = candidate_u
            last_employer_utility = employer_u

        else:
            candidate_u = 0
            employer_u = 0

        best_candidate_utility = max(best_candidate_utility, candidate_u)
        best_employer_utility = max(best_employer_utility, employer_u)

        if is_valid_package(employer_package) and is_mutually_acceptable(employer_package):
            compatible_package_seen = True

        conversation_log.append({
            "scenario_id": scenario_id,
            "condition": condition_name,
            "run_id": run_id,
            "turn": turn,
            "speaker": "Employer",
            "text": employer_text,
            **employer_package,
            "decision": parsed_employer["decision"],
            "candidate_utility": candidate_u,
            "employer_utility": employer_u
        })

        if parsed_employer["decision"] == "quit":
            outcome = "Failure"
            break

        # If Employer accepts the previous valid package
        if parsed_employer["decision"] == "accept":
            if (
                last_valid_package is not None
                and last_candidate_utility >= MIN_CANDIDATE_UTILITY
                and last_employer_utility >= MIN_EMPLOYER_UTILITY
            ):
                outcome = "Agreement"
            else:
                outcome = "AcceptedLowUtility"
            break

        current_message = employer_text

        # Candidate turn
        candidate_text = generate_response(
            candidate_prompt,
            current_message
        )

        parsed_candidate = parse_package_response(candidate_text)
        candidate_package = extract_package(parsed_candidate)

        if is_valid_package(candidate_package):
            candidate_u = compute_utility(candidate_package, candidate_utility_table)
            employer_u = compute_utility(candidate_package, employer_utility_table)

            last_valid_package = candidate_package
            last_candidate_utility = candidate_u
            last_employer_utility = employer_u

        else:
            candidate_u = 0
            employer_u = 0

        best_candidate_utility = max(best_candidate_utility, candidate_u)
        best_employer_utility = max(best_employer_utility, employer_u)

        if is_valid_package(candidate_package) and is_mutually_acceptable(candidate_package):
            compatible_package_seen = True

        conversation_log.append({
            "scenario_id": scenario_id,
            "condition": condition_name,
            "run_id": run_id,
            "turn": turn,
            "speaker": "Candidate",
            "text": candidate_text,
            **candidate_package,
            "decision": parsed_candidate["decision"],
            "candidate_utility": candidate_u,
            "employer_utility": employer_u
        })

        if parsed_candidate["decision"] == "quit":
            outcome = "Failure"
            break

        # If Candidate accepts the previous valid package
        if parsed_candidate["decision"] == "accept":
            if (
                last_valid_package is not None
                and last_candidate_utility >= MIN_CANDIDATE_UTILITY
                and last_employer_utility >= MIN_EMPLOYER_UTILITY
            ):
                outcome = "Agreement"
            else:
                outcome = "AcceptedLowUtility"
            break

        current_message = candidate_text

    if outcome is None:
        if compatible_package_seen:
            outcome = "CompatibleButUnclosed"
        else:
            outcome = "Impasse"

    outcome_row = {
        "scenario_id": scenario_id,
        "condition": condition_name,
        "run_id": run_id,
        "outcome": outcome,
        "n_turns": len(conversation_log),
        "best_candidate_utility": best_candidate_utility,
        "best_employer_utility": best_employer_utility
    }

    return conversation_log, outcome_row

In [ ]:
log, outcome = run_utility_negotiation_simulation(
    scenario_id=1,
    condition_name="cooperative",
    candidate_style="cooperative",
    employer_style="cooperative",
    run_id=0,
    max_turns=10
)

print(outcome)
pd.DataFrame(log)

{'scenario_id': 1, 'condition': 'cooperative', 'run_id': 0, 'outcome': 'AcceptedLowUtility', 'n_turns': 3, 'best_candidate_utility': 0.7272727272727273, 'best_employer_utility': 0.5909090909090909}


,scenario_id,condition,run_id,turn,speaker,text,job_description,salary,working_hours,pension_fund,leased_car,promotion_possibilities,decision,candidate_utility,employer_utility
0,1,cooperative,0,0,Employer,"{\n ""message"": ""I'm open to discussing a Proj...",Programmer,85000,9,20%,Yes,Fast,continue,0.727273,0.363636
1,1,cooperative,0,0,Candidate,"{\n ""message"": ""I appreciate your offer, but ...",Programmer,80000,8,10%,No,Moderate,continue,0.454545,0.590909
2,1,cooperative,0,1,Employer,"{\n ""message"": ""That sounds like a good compr...",Programmer,80000,8,10%,Yes,Moderate,accept,0.545455,0.500000


In [ ]:
conditions = [
    {
        "condition_name": "cooperative",
        "candidate_style": "cooperative",
        "employer_style": "cooperative"
    },
    {
        "condition_name": "competitive",
        "candidate_style": "competitive",
        "employer_style": "competitive"
    },
    {
        "condition_name": "mixed",
        "candidate_style": "competitive",
        "employer_style": "cooperative"
    }
]

In [ ]:
n_runs = 3
max_turns = 10
scenario_ids = [1, 2, 3]

all_utility_turns = []
all_utility_outcomes = []

for scenario_id in tqdm(scenario_ids):
    for condition in conditions:
        for run_id in range(n_runs):

            print(
                f"Running scenario {scenario_id} | "
                f"{condition['condition_name']} | run {run_id}"
            )

            log, outcome = run_utility_negotiation_simulation(
                scenario_id=scenario_id,
                condition_name=condition["condition_name"],
                candidate_style=condition["candidate_style"],
                employer_style=condition["employer_style"],
                run_id=run_id,
                max_turns=max_turns
            )

            all_utility_turns.extend(log)
            all_utility_outcomes.append(outcome)

            pd.DataFrame(all_utility_turns).to_csv(
                os.path.join(RESULTS_DIR, "utility_llm_turns_llama3.csv"),
                index=False
            )

            pd.DataFrame(all_utility_outcomes).to_csv(
                os.path.join(RESULTS_DIR, "utility_llm_outcomes_llama3.csv"),
                index=False
            )

utility_turns = pd.DataFrame(all_utility_turns)
utility_outcomes = pd.DataFrame(all_utility_outcomes)

utility_outcomes

  0%|          | 0/3 [00:00<?, ?it/s]

Running scenario 1 | cooperative | run 0
Running scenario 1 | cooperative | run 1
Running scenario 1 | cooperative | run 2
Running scenario 1 | competitive | run 0
Running scenario 1 | competitive | run 1
Running scenario 1 | competitive | run 2
Running scenario 1 | mixed | run 0
Running scenario 1 | mixed | run 1
Running scenario 1 | mixed | run 2


 33%|███▎      | 1/3 [01:18<02:37, 78.68s/it]

Running scenario 2 | cooperative | run 0
Running scenario 2 | cooperative | run 1
Running scenario 2 | cooperative | run 2
Running scenario 2 | competitive | run 0
Running scenario 2 | competitive | run 1
Running scenario 2 | competitive | run 2
Running scenario 2 | mixed | run 0
Running scenario 2 | mixed | run 1
Running scenario 2 | mixed | run 2


 67%|██████▋   | 2/3 [03:12<01:39, 99.08s/it]

Running scenario 3 | cooperative | run 0
Running scenario 3 | cooperative | run 1
Running scenario 3 | cooperative | run 2
Running scenario 3 | competitive | run 0
Running scenario 3 | competitive | run 1
Running scenario 3 | competitive | run 2
Running scenario 3 | mixed | run 0
Running scenario 3 | mixed | run 1
Running scenario 3 | mixed | run 2


100%|██████████| 3/3 [05:44<00:00, 114.69s/it]


,scenario_id,condition,run_id,outcome,n_turns,best_candidate_utility,best_employer_utility
0,1,cooperative,0,AcceptedLowUtility,4,0.863636,0.590909
1,1,cooperative,1,AcceptedLowUtility,5,0.727273,0.590909
2,1,cooperative,2,AcceptedLowUtility,6,0.863636,0.681818
3,1,competitive,0,AcceptedLowUtility,9,0.863636,0.454545
4,1,competitive,1,AcceptedLowUtility,2,0.545455,0.590909
5,1,competitive,2,AcceptedLowUtility,1,0.272727,0.772727
6,1,mixed,0,AcceptedLowUtility,5,0.818182,0.363636
7,1,mixed,1,AcceptedLowUtility,2,0.772727,0.454545
8,1,mixed,2,AcceptedLowUtility,4,0.772727,0.727273
9,2,cooperative,0,AcceptedLowUtility,4,0.818182,0.363636


In [ ]:
utility_outcomes["outcome"].value_counts()

,count
outcome,
AcceptedLowUtility,26
Impasse,1


In [ ]:
utility_outcomes.groupby("condition")["outcome"].value_counts()

condition    outcome           
competitive  AcceptedLowUtility    9
cooperative  AcceptedLowUtility    9
mixed        AcceptedLowUtility    8
             Impasse               1
Name: count, dtype: int64

In [ ]:
utility_outcomes.groupby("condition")[[
    "best_candidate_utility",
    "best_employer_utility"
]].mean()

,best_candidate_utility,best_employer_utility
condition,,
competitive,0.712121,0.616162
cooperative,0.782828,0.555556
mixed,0.737374,0.611111


In [ ]:
utility_turns.to_csv(
    os.path.join(RESULTS_DIR, "utility_llm_turns_llama3.csv"),
    index=False
)

utility_outcomes.to_csv(
    os.path.join(RESULTS_DIR, "utility_llm_outcomes_llama3.csv"),
    index=False
)

In [ ]:
from google.colab import files

files.download(os.path.join(RESULTS_DIR, "utility_llm_turns_llama3.csv"))
files.download(os.path.join(RESULTS_DIR, "utility_llm_outcomes_llama3.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>